In [2]:
import os
import sys
import mne

sub_id = '02'
path = f'/mnt/dataset2/Datasets/CIRE/CIRE/rawData/rawData/sub-{sub_id}/ses-01/eeg/'
name = f'sub-{sub_id}_ses-01_task-CIRE_eeg.edf'
full_path = os.path.join(path, name)
raw = mne.io.read_raw_edf(full_path, preload=True)
print(raw.info)

eeg_data = raw.get_data()
print(eeg_data.shape)
ch_names = raw.info['ch_names']
fs = raw.info['sfreq']

ImportError: cannot import name 'sph_harm' from 'scipy.special' (/home/xux/miniconda3/envs/myenv/lib/python3.12/site-packages/scipy/special/__init__.py)

In [20]:
import pandas as pd

event_file = f'sub-{sub_id}_ses-01_task-CIRE_events.tsv'
event_path = os.path.join(path, event_file)

events_df = pd.read_csv(event_path, sep='\t')
print(events_df.head())


    onset  duration  trial_type  value  sample
0  36.552       0.0         101      2   36552
1  44.555       0.0         102      3   44555
2  53.267       0.0          11      4   53267
3  55.426       0.0          10      1   55426
4  66.732       0.0          12      5   66732


In [23]:
label_names = [11, 21, 12, 22, 13, 23, 14, 24]
end_name = 10

orders = []
start_times = []
end_times = []

for index, row in events_df.iterrows():
    event_type = row['trial_type']
    if event_type in label_names:
        orders.append(event_type)
        start_times.append(row['onset'])
    elif event_type == end_name:
        end_times.append(row['onset'])
        
print("Orders:", orders)
print("Start times:", start_times)
print("End times:", end_times)

Orders: [11.0, 12.0, 21.0, 21.0, 14.0, 21.0, 22.0, 21.0, 14.0, 12.0, 11.0, 22.0, 21.0, 14.0, 22.0, 22.0, 22.0, 13.0, 12.0, 12.0, 13.0, 22.0, 22.0, 11.0, 22.0, 22.0, 21.0, 11.0, 11.0, 13.0, 21.0, 21.0, 24.0, 14.0, 22.0, 11.0, 24.0, 24.0, 21.0, 22.0, 21.0, 12.0, 21.0, 22.0, 21.0, 24.0, 22.0, 12.0, 11.0, 14.0, 11.0, 21.0, 11.0, 13.0, 11.0]
Start times: [53.267, 66.732, 81.676, 95.839, 112.344, 127.479, 142.242, 155.835, 171.055, 184.778, 224.554, 237.652, 250.784, 265.913, 278.959, 293.468, 308.036, 322.281, 335.595, 348.061, 387.641, 400.265, 414.814, 429.618, 444.776, 459.637, 473.612, 488.193, 501.287, 513.336, 550.692, 564.994, 579.312, 595.171, 609.527, 623.183, 636.157, 654.14, 666.756, 680.27, 719.971, 734.325, 748.01, 761.944, 776.303, 789.82, 806.703, 819.624, 831.365, 843.49, 878.515, 890.568, 908.136, 920.477, 935.614]
End times: [55.426, 68.84, 84.341, 98.22, 114.329, 129.815, 144.484, 158.457, 173.588, 186.585, 226.462, 239.735, 253.344, 267.775, 281.084, 295.772, 310.64, 324

In [24]:
# to trials
trials = []
for i in range(len(orders)):
    start_sample = int(start_times[i] * fs)
    end_sample = int(end_times[i] * fs)
    trial_data = eeg_data[:, start_sample:end_sample]
    trials.append(trial_data)
print(f"Total trials extracted: {len(trials)}")
# save to a .pkl file
import pickle
output_base = f'/mnt/dataset4/daily_eeg_emotion/Data/CIRE'
output_file = f'sub-{sub_id}_trials.pkl'
output_path = os.path.join(output_base, output_file)

with open(output_path, 'wb') as f:
    pickle.dump(trials, f)

Total trials extracted: 55


In [ ]:
import os
import sys
import mne
import pandas as pd
import pickle

base_path = f'/mnt/dataset2/Datasets/CIRE/CIRE/rawData/rawData'
label_names = [11, 21, 12, 22, 13, 23, 14, 24]
end_name = 10
save_base = f'/mnt/dataset4/daily_eeg_emotion/Data/CIRE'

for sub_id in range(1, 39):
    for ses in range(1, 6):
        sub_str = f'sub-{sub_id:02d}'
        ses_str = f'ses-{ses:02d}'
        eeg_path = os.path.join(base_path, sub_str, ses_str, 'eeg')
        eeg_file = f'{sub_str}_{ses_str}_task-CIRE_eeg.edf'
        raw_file_path = os.path.join(eeg_path, eeg_file)
        if not os.path.exists(raw_file_path):
            print(f"File not found: {raw_file_path}")
            continue
        raw = mne.io.read_raw_edf(raw_file_path, preload=True)
        ch_names = raw.info['ch_names']
        fs = raw.info['sfreq']
        target_fs = 200
        
        #滤波至0.5-70Hz，并降采样到200Hz
        raw.filter(0.5, 70)
        raw.resample(target_fs)
        eeg_data = raw.get_data()
        
        event_file = f'{sub_str}_{ses_str}_task-CIRE_events.tsv'
        event_path = os.path.join(eeg_path, event_file)
        if not os.path.exists(event_path):
            print(f"Event file not found: {event_path}")
            continue
        events_df = pd.read_csv(event_path, sep='\t')
        
        orders = []
        start_times = []
        end_times = []
        labels = []
        trial_datas = []
        
        for index, row in events_df.iterrows():
            event_type = row['trial_type']
            if event_type in label_names:
                orders.append(event_type)
                start_times.append(row['onset'])
            elif event_type == end_name:
                end_times.append(row['onset'])
                
        for i in range(len(orders)):
            start_sample = int(start_times[i] * target_fs)
            end_sample = int(end_times[i] * target_fs)
            trial_data = eeg_data[:, start_sample:end_sample]
            labels.append(orders[i])
            trial_datas.append(trial_data)
        
        print(f"Subject: {sub_str}, Session: {ses_str}, Trials extracted: {len(labels)}")
        trial_dict = {
            'data': trial_datas,
            'ch_names': ch_names,
            'fs': target_fs,
            'labels': labels
        }
        save_path = os.path.join(save_base, sub_str, ses_str)
        os.makedirs(save_path, exist_ok=True)
        save_file = f'{sub_str}_{ses_str}_trial_data.pkl'
        with open(os.path.join(save_path, save_file), 'wb') as f:
            pickle.dump(trial_dict, f)
                

Extracting EDF parameters from /mnt/dataset2/Datasets/CIRE/CIRE/rawData/rawData/sub-01/ses-01/eeg/sub-01_ses-01_task-CIRE_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1181999  =      0.000 ...  1181.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 70 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 70.00 Hz
- Upper transition bandwidth: 17.50 Hz (-6 dB cutoff frequency: 78.75 Hz)
- Filter length: 6601 samples (6.601 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.6s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    2.4s


Subject: sub-01, Session: ses-01, Trials extracted: 55
Extracting EDF parameters from /mnt/dataset2/Datasets/CIRE/CIRE/rawData/rawData/sub-01/ses-02/eeg/sub-01_ses-02_task-CIRE_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1030999  =      0.000 ...  1030.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 70 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 70.00 Hz
- Upper transition bandwidth: 17.50 Hz (-6 dB cutoff frequency: 78.75 Hz)
- Filter length: 6601 samples (6.601 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.6s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    2.3s


Subject: sub-01, Session: ses-02, Trials extracted: 55
Extracting EDF parameters from /mnt/dataset2/Datasets/CIRE/CIRE/rawData/rawData/sub-01/ses-03/eeg/sub-01_ses-03_task-CIRE_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1026999  =      0.000 ...  1026.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 70 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 70.00 Hz
- Upper transition bandwidth: 17.50 Hz (-6 dB cutoff frequency: 78.75 Hz)
- Filter length: 6601 samples (6.601 s)



...

Subject: sub-01, Session: ses-03, Trials extracted: 55
Extracting EDF parameters from /mnt/dataset2/Datasets/CIRE/CIRE/rawData/rawData/sub-01/ses-04/eeg/sub-01_ses-04_task-CIRE_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1017999  =      0.000 ...  1017.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 70 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 70.00 Hz
- Upper transition bandwidth: 17.50 Hz (-6 dB cutoff frequency: 78.75 Hz)
- Filter length: 6601 samples (6.601 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    1.5s


Subject: sub-01, Session: ses-04, Trials extracted: 55
Extracting EDF parameters from /mnt/dataset2/Datasets/CIRE/CIRE/rawData/rawData/sub-01/ses-05/eeg/sub-01_ses-05_task-CIRE_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1094999  =      0.000 ...  1094.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 70 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 70.00 Hz
- Upper transition bandwidth: 17.50 Hz (-6 dB cutoff frequency: 78.75 Hz)
- Filter length: 6601 samples (6.601 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.4s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    2.0s


Subject: sub-01, Session: ses-05, Trials extracted: 55


In [64]:
trial_data = np.array(trial_data)
print(trial_datas[0].shape)
print(len(trial_datas))

(122, 443)
55


In [65]:
labels = np.array(labels)
print(labels[0])

21.0


In [52]:
beh_name = f'/mnt/dataset2/Datasets/CIRE/CIRE/behavioralStatistics.csv'
beh_data = pd.read_csv(beh_name)
print(beh_data.head())
sub_01_beh = beh_data['S01']
print(sub_01_beh.values)

   serial number  S01  S02  S03  S04  S05  S06  S07  S08  S09  ...  S29  S30  \
0              1  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0   
1              2  1.0  1.0  1.0  1.0  1.0  1.0  0.0  1.0  1.0  ...  1.0  1.0   
2              3  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0   
3              4  1.0  0.0  0.0  0.0  NaN  1.0  0.0  0.0  0.0  ...  0.0  0.0   
4              5  0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  ...  0.0  0.0   

   S31  S32  S33  S34  S35  S36  S37  S38  
0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
1  0.0  0.0  0.0  0.0  0.0  1.0  0.0  1.0  
2  0.0  NaN  0.0  0.0  0.0  0.0  0.0  0.0  
3  0.0  0.0  0.0  1.0  0.0  0.0  0.0  1.0  
4  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  

[5 rows x 39 columns]
[0. 1. 0. 1. 0. 1. 0. 0. 0. 1. 1. 1. 0. 1. 0. 0. 0. 1. 0. 1. 0. 1. 0. 1.
 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1.
 0. 1. 0. 1. 1. 1. 0. 1. 0. 1. 0. 1. 0. 1. 1. 1. 0. 1. 1. 1. 0. 1. 0. 1.
 0. 1. 0. 1. 

In [1]:
import os
import sys
import mne
import pandas as pd
import pickle

base_path = f'/mnt/dataset2/Datasets/CIRE/CIRE/rawData/rawData'
label_names = [11, 21, 12, 22, 13, 23, 14, 24]
end_name = 10
save_base = f'/mnt/dataset4/daily_eeg_emotion/Data/CIRE'

for sub_id in range(1, 39):
    for ses in range(1, 6):
        sub_str = f'sub-{sub_id:02d}'
        ses_str = f'ses-{ses:02d}'
        eeg_path = os.path.join(base_path, sub_str, ses_str, 'eeg')
        eeg_file = f'{sub_str}_{ses_str}_task-CIRE_eeg.edf'
        raw_file_path = os.path.join(eeg_path, eeg_file)
        if not os.path.exists(raw_file_path):
            print(f"File not found: {raw_file_path}")
            continue
        raw = mne.io.read_raw_edf(raw_file_path, preload=True)
        ch_names = raw.info['ch_names']
        fs = raw.info['sfreq']
        target_fs = 200
        
        #滤波至0.5-70Hz，并降采样到200Hz
        raw.filter(0.5, 70)
        raw.resample(target_fs)
        eeg_data = raw.get_data()
        
        event_file = f'{sub_str}_{ses_str}_task-CIRE_events.tsv'
        event_path = os.path.join(eeg_path, event_file)
        if not os.path.exists(event_path):
            print(f"Event file not found: {event_path}")
            continue
        events_df = pd.read_csv(event_path, sep='\t')
        
        orders = []
        start_times = []
        end_times = []
        labels = []
        trial_datas = []
        
        for index, row in events_df.iterrows():
            event_type = row['trial_type']
            if event_type in label_names:
                orders.append(event_type)
                start_times.append(row['onset'])
            elif event_type == end_name:
                end_times.append(row['onset'])
                
        for i in range(len(orders)):
            start_sample = int(start_times[i] * target_fs)
            end_sample = int(end_times[i] * target_fs)
            
            # 若不足2s则取开始后2s
            if end_sample - start_sample < 2 * target_fs:
                end_sample = start_sample + 2 * target_fs

            trial_data = eeg_data[:, start_sample:end_sample]
            labels.append(orders[i])
            trial_datas.append(trial_data)
        
        print(f"Subject: {sub_str}, Session: {ses_str}, Trials extracted: {len(labels)}")
        trial_dict = {
            'data': trial_datas,
            'ch_names': ch_names,
            'fs': target_fs,
            'labels': labels
        }
        save_path = os.path.join(save_base, sub_str, ses_str)
        os.makedirs(save_path, exist_ok=True)
        save_file = f'{sub_str}_{ses_str}_trial_data.pkl'
        with open(os.path.join(save_path, save_file), 'wb') as f:
            pickle.dump(trial_dict, f)
                


import pickle
import numpy as np
import os
import h5py

def stretch_axis(arr, new_t):
    n, t = arr.shape
    # 原有的时间点坐标（0 到 t-1）
    old_indices = np.linspace(0, t - 1, t)
    # 目标时间点坐标（0 到 t-1，但采样 new_t 个点）
    new_indices = np.linspace(0, t - 1, new_t)
    
    # 对每一行进行线性插值
    # np.apply_along_axis 可以简化循环
    resized_matrix = np.apply_along_axis(
        lambda row: np.interp(new_indices, old_indices, row), 
        axis=1, 
        arr=arr
    )
    
    return resized_matrix

data_base = f'/mnt/dataset4/daily_eeg_emotion/Data/CIRE'
out_base = f'/mnt/dataset2/Processed_datasets/EEG_Bench/CIRE'

target_sfreq = 200
sample_T = 2.0 # EEG sample window timelength in s
sample_stride = 2.0 # # EEG sample window stride in s
sample_samples = int(sample_T * target_sfreq)
stride_samples = int(sample_stride * target_sfreq)

MISSING_SESSIONS = [
    (5, 2),  # sub-05 ses-02
    (32, 5)  # sub-32 ses-05
]

data_segments = []
for sub_id in range(1, 39):
    data_sub = []
    sub_str = f'sub-{sub_id:02d}'
    
    sub_h5_path = os.path.join(out_base, f'{sub_str}.h5')
    os.makedirs(out_base, exist_ok=True)
    with h5py.File(sub_h5_path, 'w') as f:
        print(sub_str)
        print(f"\n处理 {sub_str}")
        for ses in range(1, 6):
            if (sub_id, ses) in MISSING_SESSIONS:
                print(f"Skipping known missing session: {sub_str} ses-{ses:02d}")
                continue
            
            pkl_path = os.path.join(data_base, sub_str, f'ses-{ses:02d}', f'{sub_str}_ses-{ses:02d}_trial_data.pkl')
            
            if not os.path.exists(pkl_path):
                print(f"Warning: file not found {pkl_path}")
                continue

            try:
                with open(pkl_path, 'rb') as file:
                    trial_dict = pickle.load(file)
            except Exception as e:
                print(f"Error: Failed to read file {pkl_path}: {e}")
                continue
            
            eeg_data = trial_dict['data']    # (n_trials, n_channels, T)
            labels = trial_dict['labels']
            fs = trial_dict['fs']
            ch_names = trial_dict['ch_names']
            ses_grp = f.create_group(f"session{ses}")
            
            for i_trial, data_trial in enumerate(eeg_data):
                if sub_id == 17 and ses == 1 and 48 <= i_trial <= 55:
                    print(f"Skipping corrupted trial: sub-{sub_id:02d} ses-{ses:02d} trial-{i_trial}")
                    continue
                print(f'sub {sub_id} trial {i_trial}: {data_trial.shape}')
                trial_grp = ses_grp.create_group(f"trial{i_trial}")
                n_samples = data_trial.shape[-1]
                sub_trial_label = labels[i_trial]
                n_slices = (n_samples - sample_samples + stride_samples) // stride_samples
                sub_trial_label = np.array(sub_trial_label).reshape(-1, 1)  # (1, 1)
                sub_trial_label = stretch_axis(sub_trial_label, n_slices)
                for i_slice, start in enumerate(range(0, n_samples - sample_samples + 1, stride_samples)):
                    end = start + sample_samples
                    slice_data = data_trial[:, start:end]
                    slice_grp = trial_grp.create_group(f"sample{i_slice}")
                    dset = slice_grp.create_dataset('eeg', data=slice_data)
                    dset.attrs['rsFreq'] = target_sfreq
                    dset.attrs['label'] = sub_trial_label[:, i_slice]
                    dset.attrs['subject_id'] = sub_id
                    dset.attrs['trial_id'] = i_trial
                    dset.attrs['session_id'] = ses
                    dset.attrs['segment_id'] = i_slice
                    dset.attrs['time_length'] = sample_T
                    dset.attrs['dataset_name'] = 'CIRE'
                    dset.attrs['chn_name'] = ch_names
                    dset.attrs['chn_pos'] = 'None'
                    dset.attrs['chn_ori'] = 'None'
                    dset.attrs['chn_type'] = 'EEG'
                    dset.attrs['montage'] = '10_20'

Extracting EDF parameters from /mnt/dataset2/Datasets/CIRE/CIRE/rawData/rawData/sub-01/ses-01/eeg/sub-01_ses-01_task-CIRE_eeg.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1181999  =      0.000 ...  1181.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 70 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 70.00 Hz
- Upper transition bandwidth: 17.50 Hz (-6 dB cutoff frequency: 78.75 Hz)
- Filter length: 6601 samples (6.601 s)

Subject: sub-01, Session: ses-01, Trials extracted: 55
Extracting EDF parameters from /mnt/dataset2/Datasets/CIRE/CIRE/rawData/rawData/sub-01/ses-02/eeg/sub-01_ses-02_task-CIRE_eeg

KeyboardInterrupt: 

In [39]:
import h5py
 
path = r'/mnt/dataset2/Processed_datasets/EEG_Bench/SEEDIV/sub_1.h5'
with h5py.File(path, 'r') as f:
    # to see the kinds of labels
    labels = []
    for session in f.keys():
        sess_grp = f[session]
        for trial in sess_grp.keys():
            trial_grp = sess_grp[trial]
            for sample in trial_grp.keys():
                sample_dset = trial_grp[sample]
                label = sample_dset.attrs['label']
                labels.append(label[0])
    unique_labels = set(labels)
    print("Unique labels in the dataset:", unique_labels)

Unique labels in the dataset: {np.int64(0), np.int64(1), np.int64(2), np.int64(3)}


In [26]:
data

<Closed HDF5 dataset>

In [44]:
import h5py
import os

segment_count = 0
path = r'/mnt/dataset2/Processed_datasets/EEG_Bench/FACED_emo'
for sub_id in range(123):
    sub_str = f'sub_{sub_id:03d}'
    sub_h5_path = os.path.join(path, f'{sub_str}.h5')
    with h5py.File(sub_h5_path, 'r') as f:
        n_sessions = len(f.keys())
        n_trials = len(f.keys())
        for trial in range(n_trials):
            trial_grp = f[f'trial{trial}']
            n_samples = len(trial_grp.keys())
            segment_count += n_samples
    print(f"Processed {sub_str}, total segments so far: {segment_count}")

print(f"Total segments in CIRE dataset: {segment_count}")

Processed sub_000, total segments so far: 907
Processed sub_001, total segments so far: 1814
Processed sub_002, total segments so far: 2726
Processed sub_003, total segments so far: 3633
Processed sub_004, total segments so far: 4540
Processed sub_005, total segments so far: 5447
Processed sub_006, total segments so far: 6354
Processed sub_007, total segments so far: 7261
Processed sub_008, total segments so far: 8173
Processed sub_009, total segments so far: 9085
Processed sub_010, total segments so far: 9992
Processed sub_011, total segments so far: 10899
Processed sub_012, total segments so far: 11806
Processed sub_013, total segments so far: 12713
Processed sub_014, total segments so far: 13620
Processed sub_015, total segments so far: 14527
Processed sub_016, total segments so far: 15435
Processed sub_017, total segments so far: 16342
Processed sub_018, total segments so far: 17249
Processed sub_019, total segments so far: 18156
Processed sub_020, total segments so far: 19063
Proc

In [9]:
import h5py

label_name_ori = [11, 21, 12, 22, 13, 23, 14, 24]
label_name_new = [0, 1, 2, 3, 4, 5, 6, 7]
label_mapping = {orig: new for orig, new in zip(label_name_ori, label_name_new)}

#遍历每个被试的每个试次的label，检查是否有长度不是1的
for i in range(1, 39):
    sub_str = f'sub-{i:02d}'
    path = f'/mnt/dataset2/Processed_datasets/EEG_Bench/CIRE/{sub_str}.h5'
    
    with h5py.File(path, 'r+') as f:
        for session in f.keys():
            sess_grp = f[session]
            for trial in sess_grp.keys():
                trial_grp = sess_grp[trial]
                for sample in trial_grp.keys():
                    sample_dset = trial_grp[sample]
                    label = sample_dset['eeg'].attrs['label']
                    
                    # mapping
                    if label[0] in label_mapping:
                        new_label = label_mapping[label[0]]
                        sample_dset['eeg'].attrs['label'] = new_label
                    else:
                        print(f"Warning: Unrecognized label {label[0]} in {sub_str} {session} {trial} {sample}")
                        
    print(f"Processed {sub_str} for label mapping.")

Processed sub-01 for label mapping.
Processed sub-02 for label mapping.
Processed sub-03 for label mapping.
Processed sub-04 for label mapping.
Processed sub-05 for label mapping.
Processed sub-06 for label mapping.
Processed sub-07 for label mapping.
Processed sub-08 for label mapping.
Processed sub-09 for label mapping.
Processed sub-10 for label mapping.
Processed sub-11 for label mapping.
Processed sub-12 for label mapping.
Processed sub-13 for label mapping.
Processed sub-14 for label mapping.
Processed sub-15 for label mapping.
Processed sub-16 for label mapping.
Processed sub-17 for label mapping.
Processed sub-18 for label mapping.
Processed sub-19 for label mapping.
Processed sub-20 for label mapping.
Processed sub-21 for label mapping.
Processed sub-22 for label mapping.
Processed sub-23 for label mapping.
Processed sub-24 for label mapping.
Processed sub-25 for label mapping.
Processed sub-26 for label mapping.
Processed sub-27 for label mapping.
Processed sub-28 for label m

In [15]:
import h5py

for i in range(123):
    sub_str = f'sub_{i:03d}'
    path = f'/mnt/dataset2/Processed_datasets/EEG_Bench/FACED_emo/{sub_str}.h5'
    
    with h5py.File(path, 'r+') as f:
        for trial in f.keys():
            trial_grp = f[trial]
            for sample in trial_grp.keys():
                sample_dset = trial_grp[sample]
                label = sample_dset['eeg'].attrs['label']
                
                label = label[:9]
            
                #从one-hot转为一个数
                label_index = label.tolist().index(1)
                sample_dset['eeg'].attrs['label'] = label_index
    print(f"Processed {sub_str} for label conversion.")

Processed sub_000 for label conversion.
Processed sub_001 for label conversion.
Processed sub_002 for label conversion.
Processed sub_003 for label conversion.
Processed sub_004 for label conversion.
Processed sub_005 for label conversion.
Processed sub_006 for label conversion.
Processed sub_007 for label conversion.
Processed sub_008 for label conversion.
Processed sub_009 for label conversion.
Processed sub_010 for label conversion.
Processed sub_011 for label conversion.
Processed sub_012 for label conversion.
Processed sub_013 for label conversion.
Processed sub_014 for label conversion.
Processed sub_015 for label conversion.
Processed sub_016 for label conversion.
Processed sub_017 for label conversion.
Processed sub_018 for label conversion.
Processed sub_019 for label conversion.
Processed sub_020 for label conversion.
Processed sub_021 for label conversion.
Processed sub_022 for label conversion.
Processed sub_023 for label conversion.
Processed sub_024 for label conversion.


In [18]:
import h5py
import os

for i in range(123):
    sub_str = f'sub_{i:03d}'
    in_path = f'/mnt/dataset2/Processed_datasets/EEG_Bench/FACED_emo_new/{sub_str}.h5'
    out_path = f'/mnt/dataset2/Processed_datasets/EEG_Bench/FACED_emo_reg/{sub_str}.h5'
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    
    with h5py.File(in_path, 'r') as f_in, h5py.File(out_path, 'w') as f_out:
        # 先将原文件完全复制到新文件
        for trial in f_in.keys():
            trial_grp_in = f_in[trial]
            trial_grp_out = f_out.create_group(trial)
            for sample in trial_grp_in.keys():
                sample_dset_in = trial_grp_in[sample]
                sample_dset_out = trial_grp_out.create_group(sample)
                # 复制数据集
                data = sample_dset_in['eeg'][:]
                dset = sample_dset_out.create_dataset('eeg', data=data)
                # 复制属性
                for attr_key, attr_value in sample_dset_in['eeg'].attrs.items():
                    dset.attrs[attr_key] = attr_value
                    
        # 然后修改label属性
        for trial in f_out.keys():
            trial_grp = f_out[trial]
            for sample in trial_grp.keys():
                sample_dset = trial_grp[sample]
                label = sample_dset['eeg'].attrs['label']
                
                label = label[9:]
                sample_dset['eeg'].attrs['label'] = label
    print(f"Processed {sub_str} for label adjustment.")


Processed sub_000 for label adjustment.
Processed sub_001 for label adjustment.
Processed sub_002 for label adjustment.
Processed sub_003 for label adjustment.
Processed sub_004 for label adjustment.
Processed sub_005 for label adjustment.
Processed sub_006 for label adjustment.
Processed sub_007 for label adjustment.
Processed sub_008 for label adjustment.
Processed sub_009 for label adjustment.
Processed sub_010 for label adjustment.
Processed sub_011 for label adjustment.
Processed sub_012 for label adjustment.
Processed sub_013 for label adjustment.
Processed sub_014 for label adjustment.
Processed sub_015 for label adjustment.
Processed sub_016 for label adjustment.
Processed sub_017 for label adjustment.
Processed sub_018 for label adjustment.
Processed sub_019 for label adjustment.
Processed sub_020 for label adjustment.
Processed sub_021 for label adjustment.
Processed sub_022 for label adjustment.
Processed sub_023 for label adjustment.
Processed sub_024 for label adjustment.


In [14]:
import h5py
file_path = r'/mnt/dataset2/Processed_datasets/EEG_Bench/SEED-V_cls/sub_0.h5'

# check attributes
with h5py.File(file_path, 'r') as f:
    print("Attrbutes for the file:")
    for attr_key, attr_value in f.attrs.items():
        print(f"{attr_key}: {attr_value}")
    for session in f.keys():
        sess_grp = f[session]
        for trial in sess_grp.keys():
            trial_grp = sess_grp[trial]
            for sample in trial_grp.keys():
                sample_dset = trial_grp[sample]
                print(f"Attributes for {session}/{trial}/{sample}:")
                for attr_key, attr_value in sample_dset.attrs.items():
                    print(f"  {attr_key}: {attr_value}")
                break  # 只查看第一个样本的属性
            break
        break

Attrbutes for the file:
chn_name: ['FP1' 'FPZ' 'FP2' 'AF3' 'AF4' 'F7' 'F5' 'F3' 'F1' 'FZ' 'F2' 'F4' 'F6'
 'F8' 'FT7' 'FC5' 'FC3' 'FC1' 'FCZ' 'FC2' 'FC4' 'FC6' 'FT8' 'T7' 'C5' 'C3'
 'C1' 'CZ' 'C2' 'C4' 'C6' 'T8' 'TP7' 'CP5' 'CP3' 'CP1' 'CPZ' 'CP2' 'CP4'
 'CP6' 'TP8' 'P7' 'P5' 'P3' 'P1' 'PZ' 'P2' 'P4' 'P6' 'P8' 'PO7' 'PO5'
 'PO3' 'POZ' 'PO4' 'PO6' 'PO8' 'CB1' 'O1' 'OZ' 'O2' 'CB2']
dataset_name: SEEDV
downstream_task_type: classification
montage: 10_20
rsFreq: 200
subject_id: 0
task_type: emotion
Attributes for trial0/sample0/eeg:
  chn_name: ['FP1' 'FPZ' 'FP2' 'AF3' 'AF4' 'F7' 'F5' 'F3' 'F1' 'FZ' 'F2' 'F4' 'F6'
 'F8' 'FT7' 'FC5' 'FC3' 'FC1' 'FCZ' 'FC2' 'FC4' 'FC6' 'FT8' 'T7' 'C5' 'C3'
 'C1' 'CZ' 'C2' 'C4' 'C6' 'T8' 'TP7' 'CP5' 'CP3' 'CP1' 'CPZ' 'CP2' 'CP4'
 'CP6' 'TP8' 'P7' 'P5' 'P3' 'P1' 'PZ' 'P2' 'P4' 'P6' 'P8' 'PO7' 'PO5'
 'PO3' 'POZ' 'PO4' 'PO6' 'PO8' 'CB1' 'O1' 'OZ' 'O2' 'CB2']
  chn_ori: None
  chn_pos: None
  chn_type: EEG
  dataset_name: SEEDV
  label: 4
  montage: 10_20
  rsFreq

In [11]:
import h5py
file_path = r'/mnt/dataset2/Processed_datasets/EEG_Bench/SEED/sub_1.h5'

# check attributes
with h5py.File(file_path, 'r') as f:
    print("Attrbutes for the file:")
    for attr_key, attr_value in f.attrs.items():
        print(f"{attr_key}: {attr_value}")
    for session in f.keys():
        sess_grp = f[session]
        for trial in sess_grp.keys():
            trial_grp = sess_grp[trial]
            for sample in trial_grp.keys():
                sample_dset = trial_grp[sample]
                print(f"Attributes for {session}/{trial}/{sample}:")
                for attr_key, attr_value in sample_dset.attrs.items():
                    print(f"  {attr_key}: {attr_value}")
                break  # 只查看第一个样本的属性
            break
        break

Attrbutes for the file:
chn_name: ['FP1' 'FPZ' 'FP2' 'AF3' 'AF4' 'F7' 'F5' 'F3' 'F1' 'FZ' 'F2' 'F4' 'F6'
 'F8' 'FT7' 'FC5' 'FC3' 'FC1' 'FCZ' 'FC2' 'FC4' 'FC6' 'FT8' 'T7' 'C5' 'C3'
 'C1' 'CZ' 'C2' 'C4' 'C6' 'T8' 'TP7' 'CP5' 'CP3' 'CP1' 'CPZ' 'CP2' 'CP4'
 'CP6' 'TP8' 'P7' 'P5' 'P3' 'P1' 'PZ' 'P2' 'P4' 'P6' 'P8' 'PO7' 'PO5'
 'PO3' 'POZ' 'PO4' 'PO6' 'PO8' 'CB1' 'O1' 'OZ' 'O2' 'CB2']
chn_ori: None
chn_pos: None
chn_type: EEG
dataset_name: SEED_3class
downstream_task_type: classification
montage: 10_10
rsFreq: 200.0
subject_id: 1
task_type: emotion
Attributes for trial0/segment0/eeg:
  end_time: 29.0
  label: [2]
  segment_id: 0
  start_time: 27.0
  time_length: 2.0


In [ ]:
# 调整SEED-V_cls的属性，将Attributes for trial0/sample0/eeg里的chn_name、dataset_name、montage、rsFreq、subject_id调整给文件属性
# 将label由数字改成数组（比如从4改成[4]）

import h5py

for i in range(16):
    file_path = rf'/mnt/dataset2/Processed_datasets/EEG_Bench/SEED-V_cls/sub_{i}.h5'

    with h5py.File(file_path, 'r+') as f:
        # 如果有session，需要先进入session
        if 'session0' in f.keys():
            first_trial = f['session0']['trial0']['sample0']['eeg']
        else:
            first_trial = f['trial0']['sample0']['eeg']
        
        # 需要调整的属性列表
        attrs_to_move = ['chn_name', 'dataset_name', 'montage', 'rsFreq', 'subject_id']
        
        for attr in attrs_to_move:
            if attr in first_trial.attrs:
                f.attrs[attr] = first_trial.attrs[attr]
                print(f"Moved attribute {attr} to file level.")
            else:
                print(f"Attribute {attr} not found in sample dataset.")
                
        # 给文件加一个新属性task_type为emotion，加一个downstream_task_type: classification
        f.attrs['task_type'] = 'emotion'
        f.attrs['downstream_task_type'] = 'classification'
        print("Added task_type and downstream_task_type attributes to file level.")
        
        # 将label由数字改成数组（比如从4改成[4]）
        for session in f.keys():
            sess_grp = f[session]
            for trial in sess_grp.keys():
                trial_grp = sess_grp[trial]
                for sample in trial_grp.keys():
                    sample_dset = trial_grp[sample]
                    if 'label' in sample_dset.attrs:
                        label_value = sample_dset.attrs['label']
                        if not isinstance(label_value, (list, tuple)):
                            sample_dset.attrs['label'] = [label_value]
                            print(f"Converted label to list for {session}/{trial}/{sample}.")

Moved attribute chn_name to file level.
Moved attribute dataset_name to file level.
Moved attribute montage to file level.
Moved attribute rsFreq to file level.
Moved attribute subject_id to file level.
Added task_type and downstream_task_type attributes to file level.
